# NLP with TensorFlow 2 (No tf.data)
Sentiment classification example

In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt

## Dataset

In [ ]:
texts = np.array([
    "this stock is going up",
    "strong earnings and growth",
    "bullish momentum continues",
    "market looks very positive",
    "great financial results",

    "this stock is crashing",
    "very bad earnings report",
    "bearish outlook ahead",
    "market uncertainty rising",
    "poor financial performance"
])

labels = np.array([1,1,1,1,1, 0,0,0,0,0])

## Train / Validation Split

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(
    texts, labels, test_size=0.2, random_state=42
)

## Tokenization

In [ ]:
max_tokens = 1000
sequence_length = 10

vectorizer = tf.keras.layers.TextVectorization(
    max_tokens=max_tokens,
    output_mode='int',
    output_sequence_length=sequence_length
)

vectorizer.adapt(X_train)

## Inspect Vocabulary

In [ ]:
vocab = vectorizer.get_vocabulary()
print("Vocabulary size:", len(vocab))
print(vocab[:20])

## Transform Text

In [ ]:
X_train_vec = vectorizer(X_train)
X_val_vec = vectorizer(X_val)

print(X_train_vec[:2])

## Build Model

In [ ]:
embedding_dim = 16

model = tf.keras.Sequential([
    tf.keras.layers.Embedding(input_dim=max_tokens, output_dim=embedding_dim),
    tf.keras.layers.GlobalAveragePooling1D(),
    tf.keras.layers.Dense(16, activation='relu'),
    tf.keras.layers.Dense(1, activation='sigmoid')
])

## Compile

In [ ]:
model.compile(
    loss='binary_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

## Train

In [ ]:
history = model.fit(
    X_train_vec,
    y_train,
    validation_data=(X_val_vec, y_val),
    epochs=20,
    verbose=1
)

## Plot

In [ ]:
plt.plot(history.history['loss'], label='train')
plt.plot(history.history['val_loss'], label='val')
plt.legend()
plt.show()

plt.plot(history.history['accuracy'], label='train')
plt.plot(history.history['val_accuracy'], label='val')
plt.legend()
plt.show()

## Predictions

In [ ]:
test_sentences = np.array([
    "strong growth and profits",
    "very bad market conditions"
])

test_vec = vectorizer(test_sentences)
preds = model.predict(test_vec)

for s, p in zip(test_sentences, preds):
    print(f"{s} → {p[0]:.3f}")